# Week 6, Day 4: A training loop, then frontier models as pricers

**Lesson:** [week6/day4.ipynb](../../../week6/day4.ipynb), *Neural networks and LLMs*. **About 25 minutes.**

**What you practice here**

- The four stages of a PyTorch training step, in the lesson's epoch loop: the forward pass, the loss, the backward pass and the optimizer's step, after clearing the old gradients.
- A one-message prompt that asks a frontier model for nothing but a price, as the lesson's `messages_for` does.
- Calling any provider through LiteLLM's `completion` with a provider-prefixed model name, and reading the reply's text from the response.
- Passing each model's own options, such as a reasoning effort or a seed, through the same call, and wrapping a model as a pricer for the lesson's `evaluate`.

**How this notebook works.** Run the first code cell once. Then, for each part below, write it in its cell, run that cell, and run the check under it. The checks hand your code stand-ins shaped like the real client and libraries: they record what your code sends and answer with canned replies, so nothing is called and nothing is spent. Write from memory, and look back at the lesson only when you are stuck.

When every check passes, save the notebook, run the checker and push, as in every session. The tracker then marks the lesson done.

In [ ]:
# Run this cell first. It finds plan/check.py in your fork and loads this exercise's checks.
import pathlib, sys
ROOT = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "plan" / "check.py").is_file()), None)
if ROOT:
    sys.path.insert(0, str(ROOT / "plan"))
    try:
        from check import exercise
    except ImportError:
        raise ImportError("your plan/check.py is older than this notebook: run git pull, or copy the new one from the tracker page") from None
    check = exercise("week6/day4", globals())
else:  # in Colab, or outside your fork: write and try the parts here, and run the checks in your codespace
    def check(part=None):
        print("The checks are off here, as plan/check.py is not in a folder above this notebook. Run them in your codespace.")
    print("plan/check.py was not found above this notebook, so the checks are off here, as in Colab.")

## The task

The lesson works its way from a vanilla neural network to frontier models. It trains an eight-layer PyTorch network on
a bag of words, for two epochs of the four-stage training loop. Then it asks frontier models to price products with
no training at all: one user message with the product's summary, sent through LiteLLM's `completion` to OpenAI,
Anthropic, Google and xAI models in turn, each wrapped in a small function that the lesson's `evaluate` scores. Write
the loop's epoch, then the frontier pricers, for wrestling memorabilia listings, each with a `summary` in Day 2's
format and a `price`.

PyTorch is not in the checks, so `train_epoch` takes the model, the batches, the loss function and the optimizer as
parameters. The functions that call a model, `price_with` and `make_pricer`, take LiteLLM's `completion` function as
their first parameter. The checks pass stand-ins shaped like them: they record what your code does and answer with
canned values, so nothing is trained, called or spent.

## What each call should give

```python
train_epoch(model, loader, loss_function, optimizer)
# trains the model on every batch of the loader once, and returns the last batch's loss, such as 9512.3

messages_for(program)
# [{'role': 'user', 'content': 'Estimate the price of this item...\n\nTitle: WrestleMania III official program\n...'}]

price_with(completion, program)
# makes one request with the model openai/gpt-4.1-nano and the messages from messages_for(program),
# and returns the reply's text as it is: '$85.00'

gemini = make_pricer(completion, "gemini/gemini-3-pro-preview", reasoning_effort="low")
gemini(program)
# makes one request with that model, the messages from messages_for(program) and a low reasoning effort
```

In [ ]:
# The imports for this exercise. Add more here if you need them: the checks have the standard library, openai, pydantic, dotenv, numpy, bs4 and tiktoken.

### 1 · `train_epoch(model, loader, loss_function, optimizer)`

Runs one epoch of the lesson's training loop and returns the last batch's loss as a number. Put the model in training
mode once, before the first batch. Then, for each `(batch_X, batch_y)` pair that `loader` gives, clear the
optimizer's old gradients and do the lesson's four stages: the forward pass (call the model on `batch_X`), the loss
(call `loss_function` on the outputs and `batch_y`), the backward pass on that loss, and one step of the optimizer.
Go through `loader` directly, without the lesson's `tqdm` progress bar, which the checks lack. Return the last loss
as a plain Python number.

<details><summary>Hint</summary>

```python
model.train()
for batch_X, batch_y in loader:
    optimizer.zero_grad()
    outputs = model(batch_X)
    loss = loss_function(outputs, batch_y)
    loss.backward()
    optimizer.step()
return loss.item()
```

</details>

In [ ]:
def train_epoch(model, loader, loss_function, optimizer):
    # Write train_epoch here, as described above.
    raise NotImplementedError("train_epoch is not written yet")

In [ ]:
check("train_epoch")

### 2 · `messages_for(listing)`

Builds the one message that asks for a price, as the lesson's `messages_for` does. Return a list holding one user
message and no system message. Its content is your instruction, which asks for the item's price and nothing else,
then a blank line, then `listing.summary`, unchanged.

<details><summary>Hint</summary>

```python
[{"role": "user", "content": instruction + "\n\n" + listing.summary}]
```

</details>

In [ ]:
def messages_for(listing):
    # Write messages_for here, as described above.
    raise NotImplementedError("messages_for is not written yet")

In [ ]:
check("messages_for")

### 3 · `price_with(completion, listing, model="openai/gpt-4.1-nano", **options)`

Asks one model for a listing's price in one LiteLLM call and returns the reply's text. `completion` is LiteLLM's
function. Call it once, passing everything by name: the model `model`, the messages from `messages_for(listing)`,
and the `options` a model needs, such as a reasoning effort or a seed, passed on as keyword arguments, with nothing
more when there are none. Return the reply's text as it is, since the lesson's `evaluate` turns text into a number.

<details><summary>Hint</summary>

```python
response = completion(model=model, messages=messages_for(listing), **options)
return response.choices[0].message.content
```

</details>

In [ ]:
def price_with(completion, listing, model="openai/gpt-4.1-nano", **options):
    # Write price_with here, as described above.
    raise NotImplementedError("price_with is not written yet")

In [ ]:
check("price_with")

### 4 · `make_pricer(completion, model="openai/gpt-4.1-nano", **options)`

Wraps one model and its options as a pricer, the function of one item that the lesson's `evaluate` calls. It plays
the part of the lesson's `gpt_4__1_nano` and `claude_opus_4_5`, for any model. Return a function that takes a
listing and returns what `price_with` gives for it, with this `completion`, `model` and `options`. Nothing is called
until the pricer runs.

<details><summary>Hint</summary>

```python
def pricer(listing):
    return price_with(completion, listing, model, **options)
return pricer
```

</details>

In [ ]:
def make_pricer(completion, model="openai/gpt-4.1-nano", **options):
    # Write make_pricer here, as described above.
    raise NotImplementedError("make_pricer is not written yet")

In [ ]:
check("make_pricer")

## Check everything

The cell below runs all 8 checks again. When they all pass, save this notebook (Ctrl+S, or Cmd+S on a Mac), then run the checker and push, as in every session:

```bash
uv run python plan/check.py
git add -A
git commit -m "Week 6, Day 4 exercise"
git push
```

The checker runs the same checks on the saved notebook, and the tracker marks the lesson done. It reads only your imports, functions, classes and constants, so the experiments in your other cells never get in the way.

In [ ]:
check()

## Try it for real

Now run what you wrote on the real thing. Write the steps in the cell at the end; they may call a model with your key, and the checks leave them out.

A guess from `gpt-4.1-nano` costs a tiny fraction of a cent, so scoring 200 items costs a few cents; 50 guesses from `gpt-5.1` cost a little more. Both need only your OpenAI key. The optional network trains on the CPU in a minute or two, with PyTorch from the course environment.

1. Load your keys with `load_dotenv(override=True)`. Put the course's `week6` folder on the path with
   `sys.path.insert(0, str(ROOT / "week6"))`, load the lesson's data with `Item.from_hub("ed-donner/items_lite")`,
   and import the lesson's `evaluate`.
2. Print `price_with(completion, test[0])` next to `test[0].price`.
3. Score `make_pricer(completion, "openai/gpt-4.1-nano")` with `evaluate`. Name the pricer `gpt_4__1_nano` first
   (`pricer.__name__ = "gpt_4__1_nano"`), since `evaluate` titles its chart with the function's name.
4. Score a second model from the lesson with its options, such as
   `make_pricer(completion, "gpt-5.1", reasoning_effort="low", seed=42)`, on fewer items with `size=50`, as the
   lesson does for its slower models. Other providers work the same way once their keys are in your `.env`.
5. Optional, the lesson's network: make its bag of words with `HashingVectorizer`, a `DataLoader` of the training
   items and its eight-layer network, as the lesson does. Train it for 2 epochs with `train_epoch`, `nn.MSELoss()`
   and `optim.Adam`, then wrap it as a pricer, as the lesson's `neural_network` does, and score it with `evaluate`.

<details><summary>Stuck? One way to write it</summary>

```python
import sys

from dotenv import load_dotenv
from litellm import completion

load_dotenv(override=True)
sys.path.insert(0, str(ROOT / "week6"))  # the lesson's pricer package
from pricer.evaluator import evaluate
from pricer.items import Item

train, val, test = Item.from_hub("ed-donner/items_lite")
print(price_with(completion, test[0]), "for an item that costs", test[0].price)

gpt_4__1_nano = make_pricer(completion, "openai/gpt-4.1-nano")
gpt_4__1_nano.__name__ = "gpt_4__1_nano"  # evaluate titles its chart with the function's name
evaluate(gpt_4__1_nano, test)

gpt_5__1 = make_pricer(completion, "gpt-5.1", reasoning_effort="low", seed=42)
gpt_5__1.__name__ = "gpt_5__1"
evaluate(gpt_5__1, test, size=50)
```

```python
# Optional: the lesson's network, trained with your train_epoch
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import HashingVectorizer
from torch.utils.data import DataLoader, TensorDataset


class NeuralNetwork(nn.Module):
    """The lesson's eight layers: 5,000 inputs to 128, then 64 six times, then one output, with ReLU between."""

    def __init__(self, input_size):
        super().__init__()
        sizes = [input_size, 128, 64, 64, 64, 64, 64, 64]
        self.layers = nn.ModuleList(nn.Linear(a, b) for a, b in zip(sizes, sizes[1:]))
        self.output = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        for layer in self.layers:
            x = self.relu(layer(x))
        return self.output(x)


vectorizer = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
X = torch.FloatTensor(vectorizer.fit_transform([item.summary for item in train]).toarray())
y = torch.FloatTensor([item.price for item in train]).unsqueeze(1)
loader = DataLoader(TensorDataset(X, y), batch_size=64, shuffle=True)

torch.manual_seed(42)
network = NeuralNetwork(X.shape[1])
loss_function = nn.MSELoss()
optimizer = optim.Adam(network.parameters(), lr=0.001)
for epoch in range(2):
    last = train_epoch(network, loader, loss_function, optimizer)
    print(f"Epoch {epoch + 1}: the last batch's loss is {last:,.0f}")


def neural_network(item):
    network.eval()
    with torch.no_grad():
        vector = torch.FloatTensor(vectorizer.transform([item.summary]).toarray())
        return max(0, network(vector)[0].item())


evaluate(neural_network, test)
```

</details>

In [ ]:
# Try it for real: write the steps above here, in as many cells as you like. This cell may call a model with your key; the checks leave it out.